# Jawad Hassan 2230-0035
# BS AI
# CV LAB # 7
# Sir Zeeshan Aslam

In [2]:
#Python Implementation from Scratch (NumPy)
import numpy as np

# Input and target
x = np.array([[1.0], [-2.0]])
t = 1.0   # Assumed ground-truth label for this worked example

# Network parameters: 2 inputs -> 2 hidden units -> 1 output
W1 = np.array([[0.5, -0.5],
               [-1.0, 2.0]])
b1 = np.array([[0.0], [1.0]])
W2 = np.array([[2.0, 1.0]])
b2 = -1.0

# ---------- Forward propagation ----------
z1 = W1 @ x + b1
h = np.maximum(0.0, z1)          # ReLU
z2 = (W2 @ h).item() + b2

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

y = sigmoid(z2)

# Clip only for numerical stability in logarithms
EPS = 1e-12
y_safe = np.clip(y, EPS, 1.0 - EPS)
loss = -(t * np.log(y_safe) + (1 - t) * np.log(1 - y_safe))

print("z1 =", z1.ravel())
print("ReLU hidden h =", h.ravel())
print("z2 =", z2)
print("prediction y =", y)
print("BCE loss =", loss)

# ---------- Backpropagation ----------
# For sigmoid + binary cross-entropy: dL/dz2 = y - t
dz2 = y - t
dW2 = dz2 * h.T
db2 = dz2

dh = W2.T * dz2
relu_mask = (z1 > 0).astype(float)
dz1 = dh * relu_mask
dW1 = dz1 @ x.T
db1 = dz1

print("dW2 =\n", dW2)
print("db2 =", db2)
print("dW1 =\n", dW1)
print("db1 =\n", db1)

# ---------- One gradient-descent update ----------
learning_rate = 0.1
W1_new = W1 - learning_rate * dW1
b1_new = b1 - learning_rate * db1
W2_new = W2 - learning_rate * dW2
b2_new = b2 - learning_rate * db2

print("Updated W1 =\n", W1_new)
print("Updated b1 =\n", b1_new)
print("Updated W2 =", W2_new)
print("Updated b2 =", b2_new)


z1 = [ 1.5 -4. ]
ReLU hidden h = [1.5 0. ]
z2 = 2.0
prediction y = 0.8807970779778823
BCE loss = 0.12692801104297263
dW2 =
 [[-0.17880438 -0.        ]]
db2 = -0.11920292202211769
dW1 =
 [[-0.23840584  0.47681169]
 [ 0.          0.        ]]
db1 =
 [[-0.23840584]
 [-0.        ]]
Updated W1 =
 [[ 0.52384058 -0.54768117]
 [-1.          2.        ]]
Updated b1 =
 [[0.02384058]
 [1.        ]]
Updated W2 = [[2.01788044 1.        ]]
Updated b2 = -0.9880797077977882


In [3]:
# 7.1 Compare ReLU and tanh hidden activations
# Reuse z1 computed above
h_relu = np.maximum(0.0, z1)
h_tanh = np.tanh(z1)

print("Pre-activation z1:", z1.ravel())
print("ReLU output:", h_relu.ravel())
print("tanh output:", h_tanh.ravel())

# Optional: output probability using tanh hidden activations
z2_tanh = (W2 @ h_tanh).item() + b2
print("Output probability with tanh:", sigmoid(z2_tanh))

Pre-activation z1: [ 1.5 -4. ]
ReLU output: [1.5 0. ]
tanh output: [ 0.90514825 -0.9993293 ]
Output probability with tanh: 0.4528820255037602


In [5]:
import torch

# Use float64 to make the comparison with NumPy more direct
xt = torch.tensor([[1.0], [-2.0]], dtype=torch.float64)
tt = torch.tensor(1.0, dtype=torch.float64)

W1t = torch.tensor([[0.5, -0.5], [-1.0, 2.0]],
                   dtype=torch.float64, requires_grad=True)
b1t = torch.tensor([[0.0], [1.0]],
                   dtype=torch.float64, requires_grad=True)
W2t = torch.tensor([[2.0, 1.0]],
                   dtype=torch.float64, requires_grad=True)
b2t = torch.tensor(-1.0, dtype=torch.float64, requires_grad=True)

z1t = W1t @ xt + b1t
ht = torch.relu(z1t)
z2t = (W2t @ ht).squeeze() + b2t
# BCE with logits is numerically stable and combines sigmoid + BCE
loss_t = torch.nn.functional.binary_cross_entropy_with_logits(z2t, tt)
loss_t.backward()

print("Loss:", loss_t.item())
print("dW1:\n", W1t.grad)
print("db1:\n", b1t.grad)
print("dW2:\n", W2t.grad)
print("db2:", b2t.grad)

Loss: 0.1269280110429725
dW1:
 tensor([[-0.2384,  0.4768],
        [ 0.0000, -0.0000]], dtype=torch.float64)
db1:
 tensor([[-0.2384],
        [ 0.0000]], dtype=torch.float64)
dW2:
 tensor([[-0.1788,  0.0000]], dtype=torch.float64)
db2: tensor(-0.1192, dtype=torch.float64)


In [6]:
import torch
print(torch.tensor([1.0]).device)        # cpu  -- the default
print(torch.tensor([1.0], device="cuda").device)   # cuda:0

cpu
cuda:0


In [7]:
# Base parameters re-declared so section 9 runs on the original values
# even if an earlier cell overwrote something.
x = np.array([[1.0], [-2.0]])
W1 = np.array([[0.5, -0.5],
               [-1.0, 2.0]])
b1 = np.array([[0.0], [1.0]])
W2 = np.array([[2.0, 1.0]])
b2 = -1.0

EPS = 1e-12

def bce(y, t):
    """Binary cross-entropy loss."""
    y_safe = np.clip(y, EPS, 1.0 - EPS)
    return -(t * np.log(y_safe) + (1 - t) * np.log(1 - y_safe))

def forward(x, W1, b1, W2, b2, activation="relu"):
    """Returns z1, h, z2, y for the chosen hidden activation."""
    z1 = W1 @ x + b1
    h = np.maximum(0.0, z1) if activation == "relu" else np.tanh(z1)
    z2 = (W2 @ h).item() + b2
    return z1, h, z2, sigmoid(z2)

def backward(x, z1, h, y, t, W2, activation="relu"):
    """Returns dW1, db1, dW2, db2 using dL/dz2 = y - t."""
    dz2 = y - t
    dW2 = dz2 * h.T
    db2 = dz2
    dh = W2.T * dz2
    gprime = (z1 > 0).astype(float) if activation == "relu" else 1.0 - np.tanh(z1) ** 2
    dz1 = dh * gprime
    dW1 = dz1 @ x.T
    db1 = dz1
    return dW1, db1, dW2, db2

# Results collected by the experiments, rendered as the table in the last cell
results = {}          # observation-table rows
extra = {}            # findings that do not fit the four table rows

def record(setting, h, y, loss, note):
    results[setting] = {"h": np.round(h.ravel(), 6), "y": y, "loss": loss, "note": note}

print("Setup complete.")

Setup complete.


In [8]:
# Exp 1 - Replace ReLU with tanh: compare hidden activations and output probability
t = 1.0
print(f"{'activation':<12}{'h':<28}{'z2':>10}{'y':>12}{'loss':>12}")
for act in ["relu", "tanh"]:
    z1, h, z2, y = forward(x, W1, b1, W2, b2, act)
    print(f"{act:<12}{str(np.round(h.ravel(), 6)):<28}{z2:>10.6f}{y:>12.6f}{bce(y, t):>12.6f}")

print("\nPre-activation z1 is the same for both:", (W1 @ x + b1).ravel())

z1, h_relu, _, y_relu = forward(x, W1, b1, W2, b2, "relu")
_, h_tanh, _, y_tanh = forward(x, W1, b1, W2, b2, "tanh")
record("ReLU; t = 1", h_relu, y_relu, bce(y_relu, 1.0),
       "baseline; unit 2 dead (z1 = -4), predicts class 1")
record("tanh; t = 1", h_tanh, y_tanh, bce(y_tanh, 1.0),
       "unit 2 stays alive and negative; y falls below 0.5, so the class flips")

activation  h                                   z2           y        loss
relu        [1.5 0. ]                     2.000000    0.880797    0.126928
tanh        [ 0.905148 -0.999329]        -0.189033    0.452882    0.792124

Pre-activation z1 is the same for both: [ 1.5 -4. ]


In [9]:
# Exp 2 - Change the target label from t = 1 to t = 0: recalculate loss and gradients
z1, h, z2, y = forward(x, W1, b1, W2, b2, "relu")
print("Forward pass is identical for both targets: h =", h.ravel(), " y =", round(y, 6))

for t_val in [1.0, 0.0]:
    dW1, db1, dW2, db2_ = backward(x, z1, h, y, t_val, W2, "relu")
    print(f"\n--- t = {t_val} ---")
    print("loss      =", round(bce(y, t_val), 6))
    print("dz2 = y-t =", round(y - t_val, 6))
    print("dW1 =\n", dW1)
    print("db1 =", db1.ravel())
    print("dW2 =", dW2.ravel())
    print("db2 =", round(db2_, 6))

record("ReLU; t = 0", h, y, bce(y, 0.0),
       "forward pass unchanged; loss 0.1269 -> 2.1269 and every gradient flips sign")

Forward pass is identical for both targets: h = [1.5 0. ]  y = 0.880797

--- t = 1.0 ---
loss      = 0.126928
dz2 = y-t = -0.119203
dW1 =
 [[-0.23840584  0.47681169]
 [ 0.          0.        ]]
db1 = [-0.23840584 -0.        ]
dW2 = [-0.17880438 -0.        ]
db2 = -0.119203

--- t = 0.0 ---
loss      = 2.126928
dz2 = y-t = 0.880797
dW1 =
 [[ 1.76159416 -3.52318831]
 [ 0.          0.        ]]
db1 = [1.76159416 0.        ]
dW2 = [1.32119562 0.        ]
db2 = 0.880797


In [10]:
# Exp 3 - Change the learning rate to 0.01, 0.1 and 0.5: compare updates and loss
t = 1.0
z1, h, z2, y = forward(x, W1, b1, W2, b2, "relu")
dW1, db1, dW2, db2_ = backward(x, z1, h, y, t, W2, "relu")
print("Loss before any update:", round(bce(y, t), 6))

print(f"\n{'lr':>6}{'W1[0,0]':>12}{'W1[0,1]':>12}{'b1[0]':>12}{'W2[0,0]':>12}{'b2':>12}"
      f"{'new y':>12}{'new loss':>12}")
lr_after = {}
for lr in [0.01, 0.1, 0.5]:
    W1_u = W1 - lr * dW1
    b1_u = b1 - lr * db1
    W2_u = W2 - lr * dW2
    b2_u = b2 - lr * db2_
    _, _, _, y_new = forward(x, W1_u, b1_u, W2_u, b2_u, "relu")
    lr_after[lr] = (y_new, bce(y_new, t))
    print(f"{lr:>6}{W1_u[0, 0]:>12.6f}{W1_u[0, 1]:>12.6f}{b1_u[0, 0]:>12.6f}"
          f"{W2_u[0, 0]:>12.6f}{b2_u:>12.6f}{y_new:>12.6f}{bce(y_new, t):>12.6f}")

print("\nRow 2 of W1 and b1[1] never move - that unit is switched off by ReLU.")

ALT_LR = 0.5
record(f"ReLU; lr = {ALT_LR}", h, y, bce(y, 1.0),
       f"same as row 1 - lr does not touch this forward pass; after the update "
       f"y = {lr_after[ALT_LR][0]:.6f}, loss = {lr_after[ALT_LR][1]:.6f}")
extra["Exp 3 - loss after one update"] = {lr: round(float(v[1]), 6) for lr, v in lr_after.items()}

Loss before any update: 0.126928

    lr     W1[0,0]     W1[0,1]       b1[0]     W2[0,0]          b2       new y    new loss
  0.01    0.502384   -0.504768    0.002384    2.001788   -0.998808    0.884168    0.123108
   0.1    0.523841   -0.547681    0.023841    2.017880   -0.988080    0.911120    0.093081
   0.5    0.619203   -0.738406    0.119203    2.089402   -0.940399    0.975591    0.024712

Row 2 of W1 and b1[1] never move - that unit is switched off by ReLU.


In [11]:
# Exp 4 - One positive and one negative pre-activation: effect of the ReLU derivative mask
t = 1.0
z1, h, z2, y = forward(x, W1, b1, W2, b2, "relu")
relu_mask = (z1 > 0).astype(float)
tanh_gprime = 1.0 - np.tanh(z1) ** 2

print("z1            =", z1.ravel(), " (unit 1 positive, unit 2 negative)")
print("ReLU h        =", h.ravel())
print("ReLU mask g'  =", relu_mask.ravel())
print("tanh g'       =", np.round(tanh_gprime.ravel(), 6))

dW1_relu, db1_relu, _, _ = backward(x, z1, h, y, t, W2, "relu")
z1_t, h_t, _, y_t = forward(x, W1, b1, W2, b2, "tanh")
dW1_tanh, db1_tanh, _, _ = backward(x, z1_t, h_t, y_t, t, W2, "tanh")

print("\ndW1 with ReLU =\n", dW1_relu)
print("db1 with ReLU =", db1_relu.ravel())
print("\ndW1 with tanh =\n", np.round(dW1_tanh, 6))
print("db1 with tanh =", np.round(db1_tanh.ravel(), 6))
print("\nRow 2 of dW1 is exactly zero under ReLU, non-zero under tanh.")

extra["Exp 4 - ReLU mask"] = relu_mask.ravel().tolist()
extra["Exp 4 - dW1 row 2, ReLU"] = dW1_relu[1].tolist()
extra["Exp 4 - dW1 row 2, tanh"] = np.round(dW1_tanh[1], 9).tolist()

z1            = [ 1.5 -4. ]  (unit 1 positive, unit 2 negative)
ReLU h        = [1.5 0. ]
ReLU mask g'  = [1. 0.]
tanh g'       = [0.180707 0.001341]

dW1 with ReLU =
 [[-0.23840584  0.47681169]
 [ 0.          0.        ]]
db1 with ReLU = [-0.23840584 -0.        ]

dW1 with tanh =
 [[-0.197736  0.395471]
 [-0.000734  0.001467]]
db1 with tanh = [-0.197736 -0.000734]

Row 2 of dW1 is exactly zero under ReLU, non-zero under tanh.


In [12]:
# Exp 5 - Compare the manual gradients with PyTorch autograd (max absolute difference)
import torch

t = 1.0
z1, h, z2, y = forward(x, W1, b1, W2, b2, "relu")
dW1, db1, dW2, db2_ = backward(x, z1, h, y, t, W2, "relu")

xt = torch.tensor(x, dtype=torch.float64)
tt = torch.tensor(t, dtype=torch.float64)
W1t = torch.tensor(W1, dtype=torch.float64, requires_grad=True)
b1t = torch.tensor(b1, dtype=torch.float64, requires_grad=True)
W2t = torch.tensor(W2, dtype=torch.float64, requires_grad=True)
b2t = torch.tensor(b2, dtype=torch.float64, requires_grad=True)

z1t = W1t @ xt + b1t
ht = torch.relu(z1t)
z2t = (W2t @ ht).squeeze() + b2t
loss_t = torch.nn.functional.binary_cross_entropy_with_logits(z2t, tt)
loss_t.backward()

print("manual loss  =", bce(y, t))
print("autograd loss=", loss_t.item())

print(f"\n{'parameter':<12}{'max |manual - autograd|':>26}")
diffs = {}
for name, manual, auto in [("dW1", dW1, W1t.grad), ("db1", db1, b1t.grad),
                           ("dW2", dW2, W2t.grad), ("db2", np.array(db2_), b2t.grad)]:
    d = float(np.max(np.abs(manual - auto.numpy())))
    diffs[name] = d
    print(f"{name:<12}{d:>26.3e}")
print(f"\nMaximum absolute difference overall: {max(diffs.values()):.3e}")

extra["Exp 5 - per-parameter max abs diff"] = {k: f"{v:.3e}" for k, v in diffs.items()}
extra["Exp 5 - overall max abs diff"] = f"{max(diffs.values()):.3e}"

manual loss  = 0.12692801104297263
autograd loss= 0.1269280110429725

parameter      max |manual - autograd|
dW1                          0.000e+00
db1                          0.000e+00
dW2                          0.000e+00
db2                          0.000e+00

Maximum absolute difference overall: 0.000e+00


In [13]:
# Observation table, filled from the values recorded by the experiments above
ORDER = ["ReLU; t = 1", "tanh; t = 1", "ReLU; t = 0", f"ReLU; lr = {ALT_LR}"]

print(f"{'Setting':<16}{'Hidden output h':<26}{'Output prob y':>14}{'Loss':>10}  Key observation")
print("-" * 135)
for key in ORDER:
    r = results[key]
    print(f"{key:<16}{str(r['h']):<26}{r['y']:>14.6f}{r['loss']:>10.6f}  {r['note']}")

print("\nAdditional recorded findings:")
for k, v in extra.items():
    print(f"  {k}: {v}")

# Write everything to a CSV next to the notebook
import csv

with open("lab7_observation_table.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["Setting", "Hidden output h", "Output probability y", "Loss", "Key observation"])
    for key in ORDER:
        r = results[key]
        w.writerow([key, " ".join(map(str, r["h"])), f"{r['y']:.6f}", f"{r['loss']:.6f}", r["note"]])
    w.writerow([])
    for k, v in extra.items():
        w.writerow([k, v])
print("\nSaved lab7_observation_table.csv")

Setting         Hidden output h            Output prob y      Loss  Key observation
---------------------------------------------------------------------------------------------------------------------------------------
ReLU; t = 1     [1.5 0. ]                       0.880797  0.126928  baseline; unit 2 dead (z1 = -4), predicts class 1
tanh; t = 1     [ 0.905148 -0.999329]           0.452882  0.792124  unit 2 stays alive and negative; y falls below 0.5, so the class flips
ReLU; t = 0     [1.5 0. ]                       0.880797  2.126928  forward pass unchanged; loss 0.1269 -> 2.1269 and every gradient flips sign
ReLU; lr = 0.5  [1.5 0. ]                       0.880797  0.126928  same as row 1 - lr does not touch this forward pass; after the update y = 0.975591, loss = 0.024712

Additional recorded findings:
  Exp 3 - loss after one update: {0.01: 0.123108, 0.1: 0.093081, 0.5: 0.024712}
  Exp 4 - ReLU mask: [1.0, 0.0]
  Exp 4 - dW1 row 2, ReLU: [0.0, 0.0]
  Exp 4 - dW1 row 2, tanh: [-0